In [ ]:
# ═══ Palmprint: training in Python → inference in Rust (single cell) ══════════════════════════════════
# Runtime: GPU recommended (Runtime → Change runtime type → T4). Steps:
#   1 clone · 2 data (Kaggle saqibshoaibdz/palm-dataset, or your own session folders) · 3 prepare ROIs
#   4 train (SupCon + SAM, open-set split) · 5 export ONNX + model card · 6 build Rust · 7 verify-model
#   8 Rust build-db + eval on the TEST palms (MCC at the calibrated threshold) · 9 OpenAI-compatible API
#   10 Lean check of the matcher + threshold-calibration theorems
import os, subprocess, json, glob, base64, time, shutil, sys

REPO, BRANCH = "https://github.com/dhaneshkk/palm-recognition-prototype", "rust-inference"
DATA_DIRS = []          # e.g. ["/content/drive/MyDrive/palms/session1", ".../session2"]; empty → Kaggle download
EPOCHS, LOSS, TARGET_FAR = 40, "supcon", 1e-3
SYNTHETIC_SMOKE = False # True: synthetic hands instead of real data (pipeline check only, no accuracy meaning)
W = "/content/palm"

def sh(cmd, check=True, env=None):
    print(f"$ {cmd}", flush=True)
    p = subprocess.run(cmd, shell=True, text=True, env={**os.environ, **(env or {})})
    if check and p.returncode: raise SystemExit(f"failed ({p.returncode}): {cmd}")

# 1 · clone
if not os.path.isdir(W): sh(f"git clone -q --depth 1 -b {BRANCH} {REPO} {W}")
os.chdir(W)
sh("pip -q install -r training/requirements.txt")

# 2 · data
if SYNTHETIC_SMOKE:
    sh("python training/synth.py /content/synth --palms 120"); DATA_DIRS = ["/content/synth/session1", "/content/synth/session2"]
elif not DATA_DIRS:
    import kagglehub  # needs Kaggle credentials (Colab secrets KAGGLE_USERNAME / KAGGLE_KEY, or kaggle.json)
    root = kagglehub.dataset_download("saqibshoaibdz/palm-dataset")
    DATA_DIRS = sorted(d for d in glob.glob(f"{root}/**/session*", recursive=True) if os.path.isdir(d))[:2]
print("data:", DATA_DIRS)
assert len(DATA_DIRS) >= 1, "no session folders found"

# 3 · prepare (preprocess/preprocessor.py — the Rust port reproduces it bit-exactly)
sh(f"python training/prepare.py {' '.join(DATA_DIRS)} -o data/rois.npz")

# 4–5 · train with SAM, calibrate on held-out palms, export ONNX + card + parity fixtures
sh(f"python training/train.py data/rois.npz --out rust/models --epochs {EPOCHS} --loss {LOSS} --target-far {TARGET_FAR}")
card = json.load(open("rust/models/palmprint_encoder_512.card.json"))
t = card["metrics"]["test"]
print(f"\nPython, test palms ({t['palms']} unseen): EER {t['eer']:.4f} · rank-1 {t['rank1']:.4f} · "
      f"threshold {card['threshold']:.4f} → TAR {t['at_threshold']['tar']:.4f} FAR {t['at_threshold']['far']:.5f} MCC {t['at_threshold']['mcc']}")

# 6 · Rust toolchain + build (inference only from here on)
if not shutil.which("cargo") and not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
import onnxruntime
os.environ["ORT_DYLIB_PATH"] = glob.glob(os.path.dirname(onnxruntime.__file__) + "/capi/libonnxruntime.so*")[0]
sh("cd rust && cargo build -q --release")
B = f"{W}/rust/target/release/palmprint --model {W}/rust/models/palmprint_encoder_512.onnx"

# 7 · Rust loads the exact model Python exported (SHA-256 + embedding parity)
sh(f"{B} verify-model")

# 8 · Rust builds the gallery and evaluates the unseen test palms at the card's threshold
sh(f"{B} --embeddings {W}/data/emb.npy --names {W}/data/names.npy build-db {' '.join(DATA_DIRS)}")
sh(f"{B} --embeddings {W}/data/emb.npy --names {W}/data/names.npy eval --split test > {W}/data/rust_eval.json")
r = json.load(open(f"{W}/data/rust_eval.json"))["report"]
print(f"Rust, test palms: EER {r['eer']:.4f} · rank-1 {r['rank1_cross_session']} · TAR {r['at_threshold']['tar']:.4f} "
      f"FAR {r['at_threshold']['far']:.5f} MCC {r['at_threshold']['mcc']}")

# 9 · OpenAI-compatible /v1/chat/completions served by the Rust binary
import urllib.request
srv = subprocess.Popen(f"{B} --embeddings {W}/data/emb.npy --names {W}/data/names.npy serve --addr 127.0.0.1:8501".split())
for _ in range(60):  # wait until the server answers
    try: urllib.request.urlopen("http://127.0.0.1:8501/v1/models", timeout=1); break
    except Exception: time.sleep(0.5)
img = sorted(glob.glob(f"{DATA_DIRS[-1]}/*"))[0]
uri = "data:image/png;base64," + base64.b64encode(open(img, "rb").read()).decode()
req = {"model": "palmprint", "messages": [{"role": "user", "content": [{"type": "image_url", "image_url": {"url": uri}}]}]}
resp = urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8501/v1/chat/completions", json.dumps(req).encode(),
                                                     {"content-type": "application/json"})).read()
print("API:", json.loads(json.loads(resp)["choices"][0]["message"]["content"])["results"][0]["matches"][:3])
srv.terminate(); srv.wait()

# 10 · Lean 4.31 (kernel only): matcher properties + far_threshold_bound / far_le_alpha
if not glob.glob("/content/lean/*/bin/lean"):
    sh("pip -q install zstandard && mkdir -p /content/lean && curl -sSfL -o /content/lean/l.tar.zst "
       "https://github.com/leanprover/lean4/releases/download/v4.31.0/lean-4.31.0-linux.tar.zst")
    import tarfile, zstandard
    with open("/content/lean/l.tar.zst", "rb") as f, zstandard.ZstdDecompressor().stream_reader(f) as z:
        tarfile.open(fileobj=z, mode="r|").extractall("/content/lean")
lean = glob.glob("/content/lean/*/bin/lean")[0]
thms = "chosen_nonempty chosen_found_iff found_antitone found_all_above far_threshold_bound far_le_alpha".split()
open("/tmp/check.lean", "w").write(open("rust/lean/Match.lean").read() + "".join(f"\n#print axioms {x}" for x in thms))
out = subprocess.run([lean, "/tmp/check.lean"], capture_output=True, text=True)
print(out.stdout[-1500:], out.stderr[-500:])
print("LEAN", "VERIFIED" if out.returncode == 0 and "sorry" not in out.stdout else "FAILED")
